# **PAAD Notebook 0: Xenium Single-Cell & Bulk RNA-seq Preprocessing**

This notebook prepares the PAAD (Pancreatic Adenocarcinoma) Xenium dataset for SIDISH training.

### Inputs (all in `data/PAAD/`):
- `cell_feature_matrix.h5` — Xenium expression matrix (190,965 cells × 474 genes, raw counts)
- `cells.parquet` — per-cell spatial metadata (x/y centroids, transcript counts, nucleus area)
- `cell_groups.csv` — 10x-assigned cell type labels (used for annotation/validation only)
- `repository_expression.tsv` — TCGA-PAAD bulk RNA-seq (60,660 genes × 174 patients, raw counts)
- `TCGA-CDR-SupplementalTableS1.xlsx` — TCGA clinical data (OS and OS.time)

### Outputs (saved to `data/PAAD/run_default/`):
- `processed_adata.h5ad` — QC-filtered single-cell AnnData (raw counts, **no normalization**)
- `processed_bulk.csv` — bulk expression + survival, log1p-scaled, columns: `[duration, event, genes...]`

### Critical design decisions:
- **Do NOT normalize or log1p the single-cell data** — `ARCHITECTURE.forward()` applies `log(x+1)` internally on every forward pass. Pre-normalizing causes double log1p and collapses the gene expression range from 0–4.8 to 0–2.2.
- **DO apply log1p to bulk** — raw TCGA counts reach ~6.4M; without log1p the encoder sees values ~793,000× out of range, saturates, and learns on noise (macrophages get labeled high-risk instead of tumour cells).

## **Step 1: Import Libraries**

In [22]:
import os
import scipy.sparse
import scanpy as sc
import numpy as np
import pandas as pd
from SIDISH.SIDISH import preprocess

## **Step 2: Load Xenium Single-Cell Data**

We load the expression matrix from the Xenium `.h5` file (10x Genomics format) and merge per-cell spatial metadata from `cells.parquet`.

Key metadata columns added to `adata.obs`:
- `x_centroid`, `y_centroid` — spatial coordinates for graph construction
- `transcript_counts` — used for QC filtering
- `nucleus_area` — used for QC filtering

In [23]:
DATA_DIR = "../data/PAAD"

# Load Xenium expression matrix
adata = sc.read_10x_h5(os.path.join(DATA_DIR, "cell_feature_matrix.h5"))
adata.var_names_make_unique()

# Load per-cell spatial metadata and merge into adata.obs
cells_meta = pd.read_parquet(os.path.join(DATA_DIR, "cells.parquet"))
cells_meta = cells_meta.set_index("cell_id")

# Align index: Xenium barcodes match cell_id in parquet
adata.obs = adata.obs.join(cells_meta[["x_centroid", "y_centroid", "transcript_counts", "nucleus_area"]])

print("Loaded:", adata.n_obs, "cells ×", adata.n_vars, "genes")
adata

Loaded: 190965 cells × 474 genes


AnnData object with n_obs × n_vars = 190965 × 474
    obs: 'x_centroid', 'y_centroid', 'transcript_counts', 'nucleus_area'
    var: 'gene_ids', 'feature_types', 'genome'

## **Step 3: QC Filtering**

We apply the QC thresholds from the paper to match the reported 190,965 post-QC cell count:
- `transcript_counts >= 10` — remove near-empty droplets
- `transcript_counts <= 500` — remove likely doublets / damaged cells
- `nucleus_area > 0` — remove cells without a detected nucleus

We do **not** call `sc.pp.normalize_total()` or `sc.pp.log1p()` here. The VAE applies `log(x+1)` internally.

In [24]:
print("Before QC:", adata.n_obs, "cells")

adata = adata[adata.obs["transcript_counts"] >= 10].copy()
adata = adata[adata.obs["transcript_counts"] <= 500].copy()
adata = adata[adata.obs["nucleus_area"] > 0].copy()

print("After QC: ", adata.n_obs, "cells")
print("SIDISH:    ~190,965 cells")

Before QC: 190965 cells
After QC:  190273 cells
SIDISH:    ~190,965 cells


## **Step 4: Add Cell Type Annotations**

We merge the 10x-assigned cell type labels from `cell_groups.csv` into `adata.obs`. These labels are **not used during SIDISH training** — they are used only for post-hoc validation (checking that high-risk cells are enriched for tumour types).

In [25]:
cell_groups = pd.read_csv(os.path.join(DATA_DIR, "cell_groups.csv"))
cell_groups = cell_groups.set_index("cell_id")

adata.obs = adata.obs.join(cell_groups["group"], how="left")
adata.obs.rename(columns={"group": "cell_type"}, inplace=True)

print("Cell type distribution:")
print(adata.obs["cell_type"].value_counts().to_string())

Cell type distribution:
cell_type
Macrophages                    28263
Fibroblasts                    27393
Metaplastic Cells              19476
Tumor Cells                    18837
Acinar                         18667
CFTR- Tumor Cells              18558
T Cells                        12649
Endothelial                    10315
Endocrine 1                     9223
Endocrine 2                     9163
Ductal                          4547
CXCL9/10 Cells                  3439
B Cells                         3215
Mast Cells                      2954
Smooth Muscle Cells             1897
Lymphatic Endothelial Cells     1677


## **Step 5: Densify Single-Cell Data**

The VAE requires a dense numpy array — it crashes on sparse object arrays. We convert `.X` from sparse CSR to a dense float32 matrix here.

We save after `preprocess` (Step 8) so the file contains the gene-intersected version that matches the bulk feature space.

In [26]:
OUTPUT_DIR = "outputs/run_default"
os.makedirs(OUTPUT_DIR, exist_ok=True)

if scipy.sparse.issparse(adata.X):
    adata.X = adata.X.toarray()

adata.X = adata.X.astype(np.float32)

print("Densified — shape:", adata.shape)
print("X dtype:", adata.X.dtype, "| sparse:", scipy.sparse.issparse(adata.X))

Densified — shape: (190273, 474)
X dtype: float32 | sparse: False


## **Step 6: Load Bulk RNA-seq Data**

We load the TCGA-PAAD bulk RNA-seq matrix (genes as rows, patients as columns). Column names are already 12-character TCGA patient barcodes — no truncation needed here, but we enforce it for safety.

In [27]:
# Genes as rows, patients as columns
df_expr = pd.read_csv(os.path.join(DATA_DIR, "repository_expression.tsv"), sep="\t", index_col=0)

# Enforce 12-char patient IDs
df_expr.columns = [c[:12] for c in df_expr.columns]

print("Bulk expression shape (genes × patients):", df_expr.shape)
print("Raw count range: {:.0f} – {:,.0f}".format(df_expr.values.min(), df_expr.values.max()))

Bulk expression shape (genes × patients): (60660, 174)
Raw count range: 0 – 6,371,207


## **Step 7: Load Clinical Survival Data**

We extract `OS.time` (days) and `OS` (1=dead, 0=alive) from the TCGA CDR for PAAD patients that have expression data.

We keep the original column names (`OS.time`, `OS`) here — `preprocess(processed=True)` will rename them to `duration` and `event` via its `survival_` and `status` parameters.

In [28]:
clinical = pd.read_excel(
    os.path.join(DATA_DIR, "TCGA-CDR-SupplementalTableS1.xlsx"),
    sheet_name="TCGA-CDR"
)

paad = clinical[
    (clinical["type"] == "PAAD") &
    (clinical["bcr_patient_barcode"].isin(df_expr.columns))
].copy()

# Keep OS.time / OS column names — preprocess will rename them via survival_ and status params
survival_df = paad[["bcr_patient_barcode", "OS.time", "OS"]].dropna(subset=["OS.time", "OS"]).copy()
survival_df = survival_df.set_index("bcr_patient_barcode")

print("Patients with expression + survival data:", len(survival_df))
print("Events (dead=1):", int(survival_df["OS"].sum()), "| Censored (alive=0):", int((survival_df["OS"] == 0).sum()))
survival_df.head()

Patients with expression + survival data: 174
Events (dead=1): 90 | Censored (alive=0): 84


,OS.time,OS
bcr_patient_barcode,,
TCGA-2J-AAB1,66.0,1.0
TCGA-2J-AAB4,729.0,0.0
TCGA-2J-AAB6,293.0,1.0
TCGA-2J-AAB8,80.0,0.0
TCGA-2J-AAB9,627.0,1.0


## **Step 8: Apply log1p to Bulk, then call `preprocess(processed=True)`**

We apply `log1p` to the raw bulk counts first (this step is NOT done by `preprocess`), then hand off to `preprocess(processed=True)` which:
- Finds the gene intersection between bulk columns and SC gene names
- Reorders bulk gene columns to match SC gene order
- Concatenates the survival DataFrame and renames columns to `[duration, event, genes...]`
- Applies `map_event_column` to enforce strict 0/1 encoding

We pass `survival_="OS.time"` and `status="OS"` to match our column names.

In [29]:
import h5py

# Read Ensembl→symbol mapping directly from h5 features metadata (fast, idempotent on re-run)
with h5py.File(os.path.join(DATA_DIR, "cell_feature_matrix.h5"), "r") as f:
    ensembl_ids  = [x.decode() for x in f["matrix/features/id"][:]]
    gene_symbols = [x.decode() for x in f["matrix/features/name"][:]]
ensembl_to_symbol = dict(zip(ensembl_ids, gene_symbols))

# Transpose to (patients × genes) and keep only patients with survival data
bulk_raw = df_expr.T  # patients × genes, columns are Ensembl IDs
bulk_raw = bulk_raw.loc[bulk_raw.index.isin(survival_df.index)]

# Rename bulk columns Ensembl → symbol
bulk_raw.columns = [ensembl_to_symbol.get(g, g) for g in bulk_raw.columns]

# Deduplicate by symbol — two Ensembl IDs can map to the same symbol after rename
bulk_raw = bulk_raw.loc[:, ~bulk_raw.columns.duplicated(keep="first")]

# Keep only the Xenium panel genes
xenium_genes = set(adata.var_names)
bulk_raw = bulk_raw[[c for c in bulk_raw.columns if c in xenium_genes]]
print(f"After Ensembl→symbol rename + dedup: {bulk_raw.shape[1]} Xenium genes retained")

# Apply log1p before passing to preprocess — raw TCGA counts reach ~6.4M
bulk_log1p = np.log1p(bulk_raw)
print(f"After log1p — max: {bulk_log1p.values.max():.2f}  (raw was ~6.4M)")

# Drop gene_ids from adata.var so preprocess falls back to var_names (gene symbols)
if "gene_ids" in adata.var.columns:
    adata.var = adata.var.drop(columns=["gene_ids"])

# preprocess(processed=True) handles:
#   - gene intersection between bulk columns and adata.var_names
#   - reordering bulk gene columns to match SC gene order
#   - survival concat + column rename (OS.time → duration, OS → event)
adata_final, bulk_merged = preprocess(
    adata,
    bulk_log1p,
    survival_df,
    patient_id="cell_type",
    celltype_name="cell_type",
    processed=True,
    survival_="OS.time",
    status="OS"
)

print("\nFinal adata shape:", adata_final.shape)
print("Final bulk shape:", bulk_merged.shape)
print("Column order check — first 3:", bulk_merged.columns[:3].tolist())
bulk_merged.head()

After Ensembl→symbol rename + dedup: 474 Xenium genes retained
After log1p — max: 15.16  (raw was ~6.4M)

Final adata shape: (190273, 474)
Final bulk shape: (174, 476)
Column order check — first 3: ['duration', 'event', 'ABCC11']


,duration,event,ABCC11,ACE2,ACKR1,ACTA2,ACTG2,ACTN1,ADAM28,ADAMTS1,...,UMOD,UPK3B,UQCC2,VAMP8,VCAN,VSIG4,VTN,VWA5A,VWF,YAF2
TCGA-2J-AAB1,66.0,1,3.737670,6.249975,7.389564,8.433812,5.303305,9.513477,8.909641,7.796880,...,1.386294,4.317488,6.505784,9.212338,9.173884,7.509335,5.521461,7.857094,7.965198,6.214608
TCGA-2J-AAB4,729.0,0,2.944439,5.198497,6.642487,8.517793,4.736198,10.178160,8.346405,8.223091,...,1.386294,4.110874,6.816736,9.126633,10.018912,6.951772,6.909753,8.633197,9.020752,6.396930
TCGA-2J-AAB6,293.0,1,1.945910,1.791759,3.583519,9.010669,5.837730,9.800846,5.583496,7.247081,...,0.000000,6.929517,6.970730,8.419139,10.186371,7.327781,3.258097,7.099202,8.130942,6.535241
TCGA-2J-AAB8,80.0,0,6.035481,4.983607,5.627621,9.290537,7.797291,9.193601,6.754604,7.400010,...,0.000000,4.867534,6.361302,7.751475,9.801676,6.107023,2.639057,7.283448,7.583248,6.079933
TCGA-2J-AAB9,627.0,1,1.386294,3.295837,6.770789,8.596374,6.699500,9.343909,6.476972,7.291656,...,4.077537,4.442651,6.040255,8.075894,9.417923,6.664409,6.113682,6.924612,8.538172,5.442418


In [30]:
adata_final.write_h5ad(os.path.join(OUTPUT_DIR, "processed_adata.h5ad"))
bulk_merged.to_csv(os.path.join(OUTPUT_DIR, "processed_bulk.csv"))

print("Saved processed_adata.h5ad —", adata_final.shape)
print("Saved processed_bulk.csv   —", bulk_merged.shape)
print("Outputs in:", OUTPUT_DIR)

Saved processed_adata.h5ad — (190273, 474)
Saved processed_bulk.csv   — (174, 476)
Outputs in: outputs/run_default
